# Notebook 6 — Spectre de puissance PSD (Fig. 6)

Reproduction de la Figure 6 d'Erickson et al. (2011).

**4 panneaux :**
- (a) N=3  : PSD linéaire, f_k ∈ [0, 5.5] — pics discrets (régime périodique)
- (b) N=9  : PSD linéaire, f_k ∈ [0, 2]  — pics + harmoniques (quasi-périodique)
- (c) N=20 : PSD linéaire, f_k ∈ [0, 2]  — bruit large bande (chaotique)
- (d) N=20 : log-log, deux régimes de décroissance en loi de puissance

**Normalisation :** P(f_k) = |FFT(v̄_centre)|² / T_analyse  
**Signal :** vitesse du bloc central v̄_{N//2+1}(t̄)

Paramètres : γ_μ = 0.5, γ_λ = √0.2, ξ = 0.5, f̃ = 3.2, ε = 0.5


In [1]:
"""
Power spectrum of the BK N-block model — reproduction of Fig. 6
Erickson et al. (2011).

4 panneaux :
  (a) N=3  : PSD, axe linéaire, f_k 0→5.5
  (b) N=9  : PSD, axe linéaire, f_k 0→2  (2 sous-axes : zoom pic + zoom harmoniques)
  (c) N=20 : PSD, axe linéaire, f_k 0→2  (bruit large-bande)
  (d) N=20 : log-log, deux regimes de décroissance

Normalisation : P(f_k) = |FFT(v_centre)|² / T_analysis
(PSD standard, unités amplitude² · temps)
"""

import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import gc, time

# ── Parameters ────────────────────────────────────────────────────────────────
GAMMA_MU  = 0.5
GAMMA_LAM = np.sqrt(0.2)
XI        = 0.5
F_TILDE   = 3.2
EPS       = 0.5
SIGMA     = 1.0

T_TRANSIENT = 500
T_ANALYSIS  = 1000   # résolution fréquentielle = 1/1000 = 0.001 Hz
DT_SAMPLE   = 0.1    # f_max = 5 Hz

# ── RK45 Dormand–Prince ───────────────────────────────────────────────────────
_A = np.array([
    [0,         0,          0,          0,          0,           0,     0],
    [1/5,       0,          0,          0,          0,           0,     0],
    [3/40,      9/40,       0,          0,          0,           0,     0],
    [44/45,    -56/15,      32/9,       0,          0,           0,     0],
    [19372/6561,-25360/2187,64448/6561,-212/729,    0,           0,     0],
    [9017/3168, -355/33,    46732/5247, 49/176,    -5103/18656,  0,     0],
    [35/384,    0,          500/1113,   125/192,   -2187/6784,   11/84, 0],
])
_B5 = np.array([35/384,     0, 500/1113,  125/192,  -2187/6784,  11/84,       0])
_B4 = np.array([5179/57600, 0, 7571/16695, 393/640, -92097/339200, 187/2100, 1/40])
_C  = np.array([0, 1/5, 3/10, 4/5, 8/9, 1, 1])

def _rk45_step(f, t, y, h):
    k = np.empty((7, len(y)))
    k[0] = f(t, y)
    for i in range(1, 7):
        k[i] = f(t + _C[i]*h, y + h * (_A[i, :i] @ k[:i]))
    y5 = y + h * (_B5 @ k)
    return y5, np.abs(y5 - (y + h * (_B4 @ k)))

def rk45_uniform(f, t0, tf, y0, dt_out, rtol=1e-6, atol=1e-8, h_max=0.5):
    y = np.array(y0, dtype=float); t = t0; h = min(1e-2, dt_out)
    t_out = np.arange(t0 + dt_out, tf + 1e-12, dt_out)
    y_out = np.zeros((len(t_out), len(y))); i_out = 0; safety = 0.9
    while t < tf and i_out < len(t_out):
        h = min(h, tf - t, t_out[i_out] - t + 1e-12)
        if h < 1e-12: h = 1e-12
        y_new, err = _rk45_step(f, t, y, h)
        scale    = atol + rtol * np.maximum(np.abs(y), np.abs(y_new))
        err_norm = np.sqrt(np.mean((err / scale)**2))
        if err_norm <= 1.0:
            t += h; y = y_new
            while i_out < len(t_out) and t >= t_out[i_out] - 1e-12:
                y_out[i_out] = y; i_out += 1
            h = min(h * safety * (err_norm**(-0.2) if err_norm > 1e-30 else 5.), h_max)
        else:
            h = max(h * safety * err_norm**(-0.25), 1e-12)
    return t_out[:i_out], y_out[:i_out]

# ── Conditions initiales & RHS ────────────────────────────────────────────────
def make_y0(N):
    u0b   = -F_TILDE * GAMMA_LAM**2 / (XI * GAMMA_MU**2)
    x_bar = np.array([(j - 0.5) * 20.0 / N for j in range(1, N + 1)])
    return np.concatenate([u0b + np.exp(-((x_bar-10.)**2)/SIGMA**2),
                           np.zeros(N), np.zeros(N)])

def make_rhs(N):
    gm2 = GAMMA_MU**2; gl2 = GAMMA_LAM**2
    def rhs(t, y):
        u = y[:N]; v = y[N:2*N]; Th = y[2*N:]
        ul  = np.concatenate([[u[0]],  u[:-1]])
        ur  = np.concatenate([u[1:],   [u[-1]]])
        vp1 = np.maximum(v + 1., 1e-15); lv = np.log(vp1)
        dv  = gm2*(ul-2*u+ur) - gl2*u - (gm2/XI)*(F_TILDE+Th+lv)
        dTh = -vp1*(Th + (1.+EPS)*lv)
        return np.concatenate([v, dv, dTh])
    return rhs

# ── Calcul PSD ────────────────────────────────────────────────────────────────
def compute_psd(N):
    t0w = time.time()
    sol = solve_ivp(make_rhs(N), [0, T_TRANSIENT], make_y0(N),
                    method='Radau', rtol=1e-5, atol=1e-7, max_step=2.)
    y_ss = sol.y[:, -1]
    _, y_arr = rk45_uniform(make_rhs(N),
                            T_TRANSIENT, T_TRANSIENT + T_ANALYSIS,
                            y_ss, DT_SAMPLE)
    v_c = y_arr[:, N + N//2]
    n   = len(v_c)
    fft_v = np.fft.rfft(v_c - v_c.mean())
    P     = np.abs(fft_v)**2 / T_ANALYSIS
    P[0]  = 0
    freqs = np.fft.rfftfreq(n, d=DT_SAMPLE)
    f_fund = freqs[np.argmax(P)]
    P = P / P.max()
    print(f'  N={N:2d}  n={n}  wall={time.time()-t0w:.1f}s  '
          f'f_fund={f_fund:.4f} Hz  (P normalised by max)')
    return freqs, P, f_fund

# ── Plot ─────────────────────────────────────────────────────────────────────
def plot_fig6(spectra):
    fig = plt.figure(figsize=(12, 10))
    fig.suptitle(
        r'Fig. 6 — Normalized power spectra (center block)'
        '\n'
        r'Erickson et al. (2011)   [$\gamma_\mu=0.5,\ \gamma_\lambda=\sqrt{0.2},'
        r'\ \xi=0.5,\ \tilde{f}=3.2,\ \varepsilon=0.5$]',
        fontsize=10, fontweight='bold'
    )

    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.35)
    ax_a = fig.add_subplot(gs[0, 0])
    ax_c = fig.add_subplot(gs[1, 0])
    ax_d = fig.add_subplot(gs[1, 1])

    # Panneau (b) : axes brisé (deux échelles Y)
    gs_b     = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[0, 1], hspace=0.08)
    ax_b_top = fig.add_subplot(gs_b[0])
    ax_b_bot = fig.add_subplot(gs_b[1])

    COLS = {3: '#1f77b4', 9: '#e07b00', 20: '#c0392b'}
    BW   = {3: None, 9: None, 20: None}   # largeur de barre (None = auto)

    # ── (a) N=3 ──────────────────────────────────────────────────────────────
    f3, P3, ff3 = spectra[3]
    fk3  = f3 / ff3                        # normaliser : pic à f_k = 1
    mk3  = fk3 <= 5.5
    bw3  = (fk3[1] - fk3[0]) * 0.8
    ax_a.bar(fk3[mk3], P3[mk3], width=bw3, color=COLS[3], alpha=0.9, linewidth=0)
    ax_a.set_xlim(0, 5.5)
    ax_a.set_xlabel(r'$f_k$', fontsize=9)
    ax_a.set_ylabel(r'$P(f_k)$', fontsize=9)
    ax_a.set_title('Power Spectrum for Center Blockk of 3 Blockk System',
                   fontsize=8.5)
    ax_a.grid(True, alpha=0.2); ax_a.tick_params(labelsize=7)
    ax_a.set_xticks([0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5])

    # ── (b) N=9 — axe brisé ──────────────────────────────────────────────────
    f9, P9, ff9 = spectra[9]
    fk9  = f9 / ff9
    mk9  = fk9 <= 2.0
    bw9  = (fk9[1] - fk9[0]) * 0.8
    P9_max = P9.max()

    # Haut : pic principal
    ax_b_top.bar(fk9[mk9], P9[mk9], width=bw9, color=COLS[9], alpha=0.9, linewidth=0)
    ax_b_top.set_xlim(0, 2.0)
    ax_b_top.set_ylim(P9_max * 0.15, P9_max * 1.08)
    ax_b_top.set_xticklabels([]); ax_b_top.tick_params(labelsize=7)
    ax_b_top.set_ylabel(r'$P(f_k)$', fontsize=8)
    ax_b_top.set_title('Power Spectrum for Center Blockk of 9 Blockk System',
                       fontsize=8.5)
    ax_b_top.grid(True, alpha=0.2)

    # Bas : harmoniques (zoom Y)
    ax_b_bot.bar(fk9[mk9], P9[mk9], width=bw9, color=COLS[9], alpha=0.9, linewidth=0)
    ax_b_bot.set_xlim(0, 2.0)
    ax_b_bot.set_ylim(0, P9_max * 0.14)
    ax_b_bot.set_xlabel(r'$f_k$', fontsize=9)
    ax_b_bot.set_ylabel(r'$P(f_k)$', fontsize=8)
    ax_b_bot.set_xticks([0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0])
    ax_b_bot.grid(True, alpha=0.2); ax_b_bot.tick_params(labelsize=7)

    # Marques de coupure d'axe
    ax_b_top.spines['bottom'].set_visible(False)
    ax_b_bot.spines['top'].set_visible(False)
    ax_b_top.tick_params(bottom=False)
    d = 0.015
    kw = dict(color='k', clip_on=False, lw=0.9)
    for ax_, y_ in [(ax_b_top, 0), (ax_b_bot, 1)]:
        tr = ax_.transAxes
        ax_.plot((-d, +d), (y_-d, y_+d), transform=tr, **kw)
        ax_.plot((1-d, 1+d), (y_-d, y_+d), transform=tr, **kw)

    # ── (c) N=20 ─────────────────────────────────────────────────────────────
    f20, P20, ff20 = spectra[20]
    fk20 = f20 / ff20
    mk20 = fk20 <= 2.0
    bw20 = (fk20[1] - fk20[0]) * 0.8
    ax_c.bar(fk20[mk20], P20[mk20], width=bw20, color=COLS[20], alpha=0.9, linewidth=0)
    ax_c.set_xlim(0, 2.0)
    ax_c.set_xlabel(r'$f_k$', fontsize=9)
    ax_c.set_ylabel(r'$P(f_k)$', fontsize=9)
    ax_c.set_title('Power Spectrum for Center Blockk of 20 Blockk System',
                   fontsize=8.5)
    ax_c.set_xticks([0, 0.5, 1.0, 1.5, 2.0])
    ax_c.grid(True, alpha=0.2); ax_c.tick_params(labelsize=7)
    ax_c.text(0.97, 0.92, 'broad-band noise\n→ chaotic motion',
              transform=ax_c.transAxes, ha='right', va='top',
              fontsize=8, color='darkred', style='italic')

    # ── (d) N=20 log-log ─────────────────────────────────────────────────────
    # Frequencys normalisées par f_fund pour que le pic soit à f_k = 1
    mk_d = (f20 > f20[1]) & (P20 > 0)
    f_d  = f20[mk_d] / ff20
    P_d  = P20[mk_d] / P20.max()
    ax_d.scatter(f_d, P_d, s=1.2, color=COLS[20], alpha=0.5, rasterized=True)
    ax_d.set_xscale('log'); ax_d.set_yscale('log')
    ax_d.set_xlim(1e-1, 1e2); ax_d.set_ylim(1e-10, 1e0)
    ax_d.set_xlabel(r'$f_k$', fontsize=9)
    ax_d.set_ylabel(r'$P(f_k)$', fontsize=9)
    ax_d.set_title('Regimes of Decay in Power Spectrum', fontsize=8.5)
    ax_d.grid(True, which='both', alpha=0.2); ax_d.tick_params(labelsize=7)
    ax_d.text(0.20, 0.38, 'exponential\ndecay',
              transform=ax_d.transAxes, fontsize=8,
              color='darkred', style='italic', ha='center')
    ax_d.text(0.70, 0.14, 'power-law\ndecay',
              transform=ax_d.transAxes, fontsize=8,
              color='darkred', style='italic', ha='center')

    out = 'bk_power_spectrum_fig6.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig); gc.collect()
    print(f'Figure → {out}')

# ── Main ─────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print('Power spectrum BK — Fig. 6 Erickson (2011)\n')
    spectra = {}
    for N in [3, 9, 20]:
        print(f'N={N}:')
        spectra[N] = compute_psd(N)
    plot_fig6(spectra)

Power spectrum BK — Fig. 6 Erickson (2011)

N=3:
  N= 3  n=9999  wall=1.8s  f_fund=0.0460 Hz  (P normalisé par max)
N=9:
  N= 9  n=9999  wall=2.1s  f_fund=0.0480 Hz  (P normalisé par max)
N=20:
  N=20  n=9999  wall=2.4s  f_fund=0.0500 Hz  (P normalisé par max)
Figure → bk_power_spectrum_fig6.png
